In [1]:
from pathlib import Path
from collections import Counter
import re
import ast
import random
import copy
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import cohen_kappa_score, mean_absolute_error, mean_squared_error

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

In [2]:
# ============================================================
# Config
# ============================================================

PROJECT_NAME = "IELTS-Writing-Evals"

TEXT_COL = "essay"
SET_COL = "essay_set"
SCORE_COL = "domain1_score"

SEED = 42

# Model config
MAX_LEN = 512
MIN_FREQ = 2
MAX_VOCAB_SIZE = 30000

EMBED_DIM = 128
HIDDEN_DIM = 128
NUM_LAYERS = 1
DROPOUT = 0.4

# Training config
BATCH_SIZE = 32
EPOCHS = 15
PATIENCE = 3
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-5
GRAD_CLIP = 1.0

METHOD_NAME = "BiLSTM-Attention Regression"

RESULT_FILENAME = "experiment2_bilstm_attention_results.csv"
PREDICTION_FILENAME = "experiment2_bilstm_attention_predictions.csv"


def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cpu


In [3]:
# ============================================================
# Find project root
# ============================================================

def find_project_root(project_name=PROJECT_NAME):
    current = Path.cwd().resolve()

    for path in [current, *current.parents]:
        if path.name == project_name:
            return path

    for path in [current, *current.parents]:
        candidate = path / project_name
        if candidate.exists() and candidate.is_dir():
            return candidate

    raise FileNotFoundError(
        f"Cannot find project root: {project_name}. "
        f"Current working directory is: {current}"
    )


PROJECT_ROOT = find_project_root()

DATA_DIR = PROJECT_ROOT / "dataset" / "asap"
RESULT_DIR = PROJECT_ROOT / "benchmark_results" / "ASAP"
RESULT_DIR.mkdir(parents=True, exist_ok=True)

train_path = DATA_DIR / "asap_train.csv"
val_path = DATA_DIR / "asap_val.csv"
test_path = DATA_DIR / "asap_test.csv"

result_path = RESULT_DIR / RESULT_FILENAME
prediction_path = RESULT_DIR / PREDICTION_FILENAME

print("Project root:", PROJECT_ROOT)
print("Train path:", train_path)
print("Val path:  ", val_path)
print("Test path: ", test_path)
print("Result path:", result_path)
print("Prediction path:", prediction_path)

for path in [train_path, val_path, test_path]:
    if not path.exists():
        raise FileNotFoundError(f"File not found: {path}")

Project root: D:\CS222\ielts\IELTS-Writing-Evals
Train path: D:\CS222\ielts\IELTS-Writing-Evals\dataset\asap\asap_train.csv
Val path:   D:\CS222\ielts\IELTS-Writing-Evals\dataset\asap\asap_val.csv
Test path:  D:\CS222\ielts\IELTS-Writing-Evals\dataset\asap\asap_test.csv
Result path: D:\CS222\ielts\IELTS-Writing-Evals\benchmark_results\ASAP\experiment2_bilstm_attention_results.csv
Prediction path: D:\CS222\ielts\IELTS-Writing-Evals\benchmark_results\ASAP\experiment2_bilstm_attention_predictions.csv


In [4]:
# ============================================================
# Load data
# ============================================================

train_df = pd.read_csv(train_path)
val_df = pd.read_csv(val_path)
test_df = pd.read_csv(test_path)

required_cols = {TEXT_COL, SET_COL, SCORE_COL}

for name, df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    missing_cols = required_cols - set(df.columns)
    if missing_cols:
        raise ValueError(f"{name}.csv missing columns: {missing_cols}")

print("Train:", train_df.shape)
print("Val:  ", val_df.shape)
print("Test: ", test_df.shape)

print("\nEssay sets in train:", sorted(train_df[SET_COL].unique()))
print("Essay sets in val:  ", sorted(val_df[SET_COL].unique()))
print("Essay sets in test: ", sorted(test_df[SET_COL].unique()))

train_df.head()

Train: (9084, 5)
Val:   (1296, 5)
Test:  (2596, 5)

Essay sets in train: [1, 2, 3, 4, 5, 6, 7, 8]
Essay sets in val:   [1, 2, 3, 4, 5, 6, 7, 8]
Essay sets in test:  [1, 2, 3, 4, 5, 6, 7, 8]


,essay_id,essay_set,essay,domain1_score,domain1_score_norm
0,14876,6,In the excerpt The Mooring Mast by Marcia Amid...,3.0,0.75
1,9985,4,The author concluded this sentence because he ...,0.0,0.00
2,3231,2,"I can think of several books that, I would not...",4.0,0.60
3,21137,8,My best friend @PERSON2 turned thirteen on a b...,39.0,0.65
4,17919,7,A time that I was patient is when I broke my f...,23.0,0.77


In [5]:
# ============================================================
# Tokenization and vocabulary
# ============================================================

PAD_TOKEN = "<PAD>"
UNK_TOKEN = "<UNK>"

PAD_IDX = 0
UNK_IDX = 1


def tokenize(text):
    """
    Simple word tokenizer.
    Phù hợp làm baseline NN truyền thống.
    """
    text = str(text).lower()
    tokens = re.findall(r"[a-zA-Z]+(?:'[a-z]+)?|\d+|[^\w\s]", text)
    return tokens


def build_vocab(texts, min_freq=MIN_FREQ, max_vocab_size=MAX_VOCAB_SIZE):
    counter = Counter()

    for text in texts:
        counter.update(tokenize(text))

    vocab = {
        PAD_TOKEN: PAD_IDX,
        UNK_TOKEN: UNK_IDX,
    }

    for token, freq in counter.most_common(max_vocab_size - 2):
        if freq >= min_freq:
            vocab[token] = len(vocab)

    return vocab


def encode_text(text, vocab, max_len=MAX_LEN):
    tokens = tokenize(text)
    ids = [vocab.get(tok, UNK_IDX) for tok in tokens]

    if len(ids) > max_len:
        ids = ids[:max_len]

    attention_mask = [1] * len(ids)

    while len(ids) < max_len:
        ids.append(PAD_IDX)
        attention_mask.append(0)

    return ids, attention_mask

In [6]:
# ============================================================
# Dataset
# ============================================================

class EssayDataset(Dataset):
    def __init__(self, df, vocab, min_score, max_score):
        self.df = df.reset_index(drop=True)
        self.vocab = vocab
        self.min_score = min_score
        self.max_score = max_score

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        input_ids, attention_mask = encode_text(
            row[TEXT_COL],
            self.vocab,
            max_len=MAX_LEN,
        )

        score = float(row[SCORE_COL])

        # Normalize score về [0, 1] để train ổn định hơn
        if self.max_score > self.min_score:
            score_norm = (score - self.min_score) / (self.max_score - self.min_score)
        else:
            score_norm = 0.0

        return {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "attention_mask": torch.tensor(attention_mask, dtype=torch.float),
            "score": torch.tensor(score, dtype=torch.float),
            "score_norm": torch.tensor(score_norm, dtype=torch.float),
        }


def create_dataloader(df, vocab, min_score, max_score, batch_size=BATCH_SIZE, shuffle=False):
    dataset = EssayDataset(
        df=df,
        vocab=vocab,
        min_score=min_score,
        max_score=max_score,
    )

    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=0,
    )

In [7]:
# ============================================================
# BiLSTM-Attention Regression Model
# ============================================================

class BiLSTMAttentionRegressor(nn.Module):
    def __init__(
        self,
        vocab_size,
        embed_dim=EMBED_DIM,
        hidden_dim=HIDDEN_DIM,
        num_layers=NUM_LAYERS,
        dropout=DROPOUT,
        pad_idx=PAD_IDX,
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embed_dim,
            padding_idx=pad_idx,
        )

        self.bilstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )

        self.attention = nn.Linear(hidden_dim * 2, 1)

        self.dropout = nn.Dropout(dropout)

        self.regressor = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1),
            nn.Sigmoid(),  # output normalized score in [0, 1]
        )

    def forward(self, input_ids, attention_mask):
        embedded = self.embedding(input_ids)

        lstm_out, _ = self.bilstm(embedded)

        # Attention score: [batch, seq_len]
        attn_scores = self.attention(lstm_out).squeeze(-1)

        # Mask padding positions
        attn_scores = attn_scores.masked_fill(attention_mask == 0, -1e9)

        attn_weights = torch.softmax(attn_scores, dim=1)

        # Weighted sum
        context = torch.sum(lstm_out * attn_weights.unsqueeze(-1), dim=1)

        context = self.dropout(context)

        output = self.regressor(context).squeeze(-1)

        return output, attn_weights

In [8]:
# ============================================================
# Metrics and train/eval helpers
# ============================================================

def denormalize_predictions(pred_norm, min_score, max_score):
    pred_norm = np.asarray(pred_norm)

    pred_score = pred_norm * (max_score - min_score) + min_score

    return pred_score


def clip_and_round_predictions(y_pred, min_score, max_score):
    y_pred = np.asarray(y_pred)
    y_pred = np.clip(y_pred, min_score, max_score)
    y_pred = np.rint(y_pred).astype(int)

    return y_pred


def compute_metrics(y_true, y_pred_rounded):
    qwk = cohen_kappa_score(y_true, y_pred_rounded, weights="quadratic")
    mae = mean_absolute_error(y_true, y_pred_rounded)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred_rounded))

    return {
        "qwk": qwk,
        "mae": mae,
        "rmse": rmse,
    }


def train_one_epoch(model, dataloader, optimizer, criterion):
    model.train()

    total_loss = 0.0

    for batch in dataloader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        score_norm = batch["score_norm"].to(device)

        optimizer.zero_grad()

        pred_norm, _ = model(input_ids, attention_mask)

        loss = criterion(pred_norm, score_norm)

        loss.backward()

        nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)

        optimizer.step()

        total_loss += loss.item() * input_ids.size(0)

    return total_loss / len(dataloader.dataset)


@torch.no_grad()
def evaluate_model(model, dataloader, min_score, max_score):
    model.eval()

    all_true = []
    all_pred_norm = []

    total_loss = 0.0
    criterion = nn.MSELoss()

    for batch in dataloader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        score = batch["score"].cpu().numpy()
        score_norm = batch["score_norm"].to(device)

        pred_norm, _ = model(input_ids, attention_mask)

        loss = criterion(pred_norm, score_norm)
        total_loss += loss.item() * input_ids.size(0)

        all_true.extend(score.tolist())
        all_pred_norm.extend(pred_norm.detach().cpu().numpy().tolist())

    y_true = np.asarray(all_true).astype(int)

    y_pred_raw = denormalize_predictions(
        all_pred_norm,
        min_score=min_score,
        max_score=max_score,
    )

    y_pred_rounded = clip_and_round_predictions(
        y_pred_raw,
        min_score=min_score,
        max_score=max_score,
    )

    metrics = compute_metrics(y_true, y_pred_rounded)
    avg_loss = total_loss / len(dataloader.dataset)

    return avg_loss, metrics, y_true, y_pred_raw, y_pred_rounded

In [ ]:
# ============================================================
# Train per essay_set
# ============================================================

results = []
all_predictions = []

essay_sets = sorted(train_df[SET_COL].unique())

start_time = time.time()

for essay_set in essay_sets:
    print(f"\n================ Essay set {essay_set} ================")

    set_seed(SEED + int(essay_set))

    train_set_df = train_df[train_df[SET_COL] == essay_set].copy()
    val_set_df = val_df[val_df[SET_COL] == essay_set].copy()
    test_set_df = test_df[test_df[SET_COL] == essay_set].copy()

    train_val_set_df = pd.concat([train_set_df, val_set_df], axis=0)

    # Không dùng test để chọn score range
    min_score = int(train_val_set_df[SCORE_COL].min())
    max_score = int(train_val_set_df[SCORE_COL].max())

    print("Train size:", len(train_set_df))
    print("Val size:  ", len(val_set_df))
    print("Test size: ", len(test_set_df))
    print("Score range:", min_score, "-", max_score)

    # Build vocab trên train set בלבד
    vocab = build_vocab(
        texts=train_set_df[TEXT_COL].astype(str).tolist(),
        min_freq=MIN_FREQ,
        max_vocab_size=MAX_VOCAB_SIZE,
    )

    print("Vocab size:", len(vocab))

    train_loader = create_dataloader(
        df=train_set_df,
        vocab=vocab,
        min_score=min_score,
        max_score=max_score,
        batch_size=BATCH_SIZE,
        shuffle=True,
    )

    val_loader = create_dataloader(
        df=val_set_df,
        vocab=vocab,
        min_score=min_score,
        max_score=max_score,
        batch_size=BATCH_SIZE,
        shuffle=False,
    )

    test_loader = create_dataloader(
        df=test_set_df,
        vocab=vocab,
        min_score=min_score,
        max_score=max_score,
        batch_size=BATCH_SIZE,
        shuffle=False,
    )

    model = BiLSTMAttentionRegressor(
        vocab_size=len(vocab),
        embed_dim=EMBED_DIM,
        hidden_dim=HIDDEN_DIM,
        num_layers=NUM_LAYERS,
        dropout=DROPOUT,
        pad_idx=PAD_IDX,
    ).to(device)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )

    criterion = nn.MSELoss()

    best_val_qwk = -999
    best_epoch = -1
    best_state_dict = None
    best_val_metrics = None
    best_val_loss = None
    patience_counter = 0

    for epoch in range(1, EPOCHS + 1):
        train_loss = train_one_epoch(
            model=model,
            dataloader=train_loader,
            optimizer=optimizer,
            criterion=criterion,
        )

        val_loss, val_metrics, _, _, _ = evaluate_model(
            model=model,
            dataloader=val_loader,
            min_score=min_score,
            max_score=max_score,
        )

        print(
            f"Epoch {epoch:02d} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Val Loss: {val_loss:.4f} | "
            f"Val QWK: {val_metrics['qwk']:.4f} | "
            f"Val MAE: {val_metrics['mae']:.4f} | "
            f"Val RMSE: {val_metrics['rmse']:.4f}"
        )

        if val_metrics["qwk"] > best_val_qwk:
            best_val_qwk = val_metrics["qwk"]
            best_epoch = epoch
            best_state_dict = copy.deepcopy(model.state_dict())
            best_val_metrics = val_metrics
            best_val_loss = val_loss
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= PATIENCE:
            print(f"Early stopping at epoch {epoch}.")
            break

    # Load best checkpoint
    model.load_state_dict(best_state_dict)

    test_loss, test_metrics, y_true, y_pred_raw, y_pred_rounded = evaluate_model(
        model=model,
        dataloader=test_loader,
        min_score=min_score,
        max_score=max_score,
    )

    print(
        f"\nBest epoch: {best_epoch} | "
        f"Test QWK: {test_metrics['qwk']:.4f} | "
        f"Test MAE: {test_metrics['mae']:.4f} | "
        f"Test RMSE: {test_metrics['rmse']:.4f}"
    )

    results.append({
        "essay_set": essay_set,
        "method": METHOD_NAME,
        "best_epoch": best_epoch,
        "score_min": min_score,
        "score_max": max_score,
        "vocab_size": len(vocab),
        "train_size": len(train_set_df),
        "val_size": len(val_set_df),
        "test_size": len(test_set_df),
        "val_qwk": best_val_metrics["qwk"],
        "val_mae": best_val_metrics["mae"],
        "val_rmse": best_val_metrics["rmse"],
        "test_qwk": test_metrics["qwk"],
        "test_mae": test_metrics["mae"],
        "test_rmse": test_metrics["rmse"],
    })

    pred_df = test_set_df.copy()
    pred_df["method"] = METHOD_NAME
    pred_df["prediction_raw"] = y_pred_raw
    pred_df["prediction_rounded"] = y_pred_rounded
    pred_df["score_min"] = min_score
    pred_df["score_max"] = max_score
    pred_df["best_epoch"] = best_epoch

    all_predictions.append(pred_df)

end_time = time.time()

print("\nTotal training time:", round((end_time - start_time) / 60, 2), "minutes")


================ Essay set 1 ================
Train size: 1248
Val size:   180
Test size:  355
Score range: 2 - 12
Vocab size: 6025


C:\Users\WINDOWS\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\triton\windows_utils.py:404: UserWarning: Failed to find CUDA.
  warnings.warn("Failed to find CUDA.")


Epoch 01 | Train Loss: 0.0268 | Val Loss: 0.0208 | Val QWK: 0.2012 | Val MAE: 1.0500 | Val RMSE: 1.4701
Epoch 02 | Train Loss: 0.0191 | Val Loss: 0.0164 | Val QWK: 0.4956 | Val MAE: 0.9722 | Val RMSE: 1.3187
Epoch 03 | Train Loss: 0.0154 | Val Loss: 0.0146 | Val QWK: 0.5842 | Val MAE: 0.9056 | Val RMSE: 1.2494
Epoch 04 | Train Loss: 0.0133 | Val Loss: 0.0153 | Val QWK: 0.5814 | Val MAE: 0.9389 | Val RMSE: 1.2758
Epoch 05 | Train Loss: 0.0114 | Val Loss: 0.0147 | Val QWK: 0.6536 | Val MAE: 0.9222 | Val RMSE: 1.2338
Epoch 06 | Train Loss: 0.0101 | Val Loss: 0.0160 | Val QWK: 0.5962 | Val MAE: 0.9444 | Val RMSE: 1.2605
Epoch 07 | Train Loss: 0.0082 | Val Loss: 0.0131 | Val QWK: 0.6775 | Val MAE: 0.8333 | Val RMSE: 1.1690
Epoch 08 | Train Loss: 0.0074 | Val Loss: 0.0127 | Val QWK: 0.6925 | Val MAE: 0.8444 | Val RMSE: 1.1402
Epoch 09 | Train Loss: 0.0059 | Val Loss: 0.0117 | Val QWK: 0.7077 | Val MAE: 0.8056 | Val RMSE: 1.0980
Epoch 10 | Train Loss: 0.0043 | Val Loss: 0.0112 | Val QWK: 0.71

In [ ]:
# ============================================================
# Save results
# ============================================================

results_df = pd.DataFrame(results)

average_row = {
    "essay_set": "average",
    "method": METHOD_NAME,
    "best_epoch": "-",
    "score_min": "-",
    "score_max": "-",
    "vocab_size": "-",
    "train_size": results_df["train_size"].sum(),
    "val_size": results_df["val_size"].sum(),
    "test_size": results_df["test_size"].sum(),
    "val_qwk": results_df["val_qwk"].mean(),
    "val_mae": results_df["val_mae"].mean(),
    "val_rmse": results_df["val_rmse"].mean(),
    "test_qwk": results_df["test_qwk"].mean(),
    "test_mae": results_df["test_mae"].mean(),
    "test_rmse": results_df["test_rmse"].mean(),
}

final_results_df = pd.concat(
    [results_df, pd.DataFrame([average_row])],
    axis=0,
    ignore_index=True,
)

predictions_df = pd.concat(all_predictions, axis=0, ignore_index=True)

final_results_df.to_csv(result_path, index=False)
predictions_df.to_csv(prediction_path, index=False)

print("Saved results to:")
print(result_path)

print("\nSaved predictions to:")
print(prediction_path)

display(final_results_df)

In [ ]:
# ============================================================
# Plot results
# ============================================================

plot_df = final_results_df[final_results_df["essay_set"].astype(str) != "average"].copy()
avg_df = final_results_df[final_results_df["essay_set"].astype(str) == "average"].copy()

plot_df["essay_set"] = plot_df["essay_set"].astype(int)


def plot_single_method_metric_by_essay_set(
    df,
    metric_col,
    ylabel,
    title,
):
    essay_sets = sorted(df["essay_set"].unique())
    method_name = df["method"].iloc[0]

    values = df.sort_values("essay_set")[metric_col].values

    x = np.arange(len(essay_sets))

    plt.figure(figsize=(12, 6))

    bars = plt.bar(
        x,
        values,
        width=0.6,
        label=method_name,
    )

    for bar in bars:
        height = bar.get_height()
        plt.text(
            bar.get_x() + bar.get_width() / 2,
            height,
            f"{height:.3f}",
            ha="center",
            va="bottom",
            fontsize=8,
        )

    plt.xticks(x, essay_sets)
    plt.xlabel("Essay Set")
    plt.ylabel(ylabel)
    plt.title(title)
    plt.legend()
    plt.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()


plot_single_method_metric_by_essay_set(
    df=plot_df,
    metric_col="test_qwk",
    ylabel="Test QWK",
    title="BiLSTM-Attention Regression: Test QWK by Essay Set",
)

plot_single_method_metric_by_essay_set(
    df=plot_df,
    metric_col="test_mae",
    ylabel="Test MAE",
    title="BiLSTM-Attention Regression: Test MAE by Essay Set",
)

plot_single_method_metric_by_essay_set(
    df=plot_df,
    metric_col="test_rmse",
    ylabel="Test RMSE",
    title="BiLSTM-Attention Regression: Test RMSE by Essay Set",
)


summary_cols = [
    "method",
    "test_qwk",
    "test_mae",
    "test_rmse",
]

avg_summary = avg_df[summary_cols].copy()

avg_summary = avg_summary.rename(columns={
    "method": "Method",
    "test_qwk": "Avg Test QWK",
    "test_mae": "Avg Test MAE",
    "test_rmse": "Avg Test RMSE",
})

avg_summary["Avg Test QWK"] = avg_summary["Avg Test QWK"].astype(float).round(4)
avg_summary["Avg Test MAE"] = avg_summary["Avg Test MAE"].astype(float).round(4)
avg_summary["Avg Test RMSE"] = avg_summary["Avg Test RMSE"].astype(float).round(4)

print("\nAverage summary:")
display(avg_summary)